# Unificación de Datos para el TFM
Este notebook unifica **14 fuentes de datos** independientes en un único dataset maestro (`dataset_final_tfm.csv`) estructurado a nivel **trimestral** y **provincial** (52 provincias × 20 años ≈ 4.368 filas).

---

## 1. Fuentes de Datos Utilizadas

| # | Archivo | Tipo | Fuente | Variable(s) generada(s) | Periodicidad original |
|---|---------|------|--------|-------------------------|-----------------------|
| 1 | `tinsa_prov_precios_trim.csv` | CSV | Tinsa (IMIE) | `Precio_m2` | Trimestral / Prov. |
| 2 | `euribor_trim_fixed.csv` | CSV | Banco Central Europeo (API) | `Euribor` | Trimestral / Nacional |
| 3 | `tasa_ahorro_trimestral_nac.csv` | CSV | INE / Contabilidad Nacional | `tasa_ahorro` | Trimestral / Nacional |
| 4 | `spread_hipotecas_trim.csv` | CSV | BdE / Tipos hipotecarios | `Spread_Hipotecas` | Trimestral / Nacional |
| 5 | `hipotecas_por_prov_men.csv` | CSV | INE — Estadística de Hipotecas | `Hipotecas_Total` | Mensual / Prov. |
| 6 | `poblacion_por_prov_combinado.csv` | CSV | INE — Padrón Municipal | `Poblacion` | Anual / Prov. |
| 7 | `edad_media_provincias_an.csv` | CSV | INE — Estadística del Padrón | `Edad_Media` | Anual / Prov. |
| 8 | `pib_pc_prov_an.xlsx` | XLSX | INE — Contabilidad Regional | `PIB_pc` | Anual / Prov. |
| 9 | `indice_mano_obra_men.csv` | CSV | Ministerio de Vivienda | `Indice_Mano_Obra` | Mensual / Nacional |
| 10 | `indice_precio_materiales_men.csv` | CSV | Ministerio de Vivienda | `Indice_Materiales` | Mensual / Nacional |
| 11 | `ipc_ccaa_men.csv` | CSV | INE — IPC por CCAA | `IPC` | Mensual / CCAA |
| 12 | `epa_prov.csv` | CSV | INE — Encuesta de Población Activa | `Tasa_Paro` | Trimestral / Prov. |
| 13 | `pob_ext_provincias_2005_2025.csv` | CSV | INE — Padrón Municipal (extranjeros) | `Pob_Ext` | Anual / Prov. |
| 14 | `transmision_prov_trim_tot.csv` | CSV | Ministerio de Vivienda / Notariado | `Transmisiones_Total` | Trimestral / Prov. |

---

## 2. Metodología de Unificación

### 2.1 Estandarización temporal
Todas las fuentes se convierten a una **base trimestral ISO** con el primer día de cada trimestre como clave temporal (ej. `2005-01-01`, `2005-04-01`…). Según la granularidad original:
- **Datos mensuales** (`hipotecas`, `IPC`, `materiales`, `mano de obra`): se asignan al trimestre correspondiente y se **agregan por media o suma**.
- **Datos anuales** (`población`, `edad media`, `PIB`, `población extranjera`): se **replican a los 4 trimestres** del año con `expandir_anual_a_trimestral`.
- **Datos ya trimestrales** (`Tinsa`, `Euríbor`, `Spread`, `Transmisiones`, `EPA`): se usan directamente.

### 2.2 Estandarización geográfica
Se define una lista canónica de las **52 provincias españolas**. Para cada fuente se aplica la función `limpiar_provincia`, que:
1. Elimina prefijos numéricos del INE.
2. Resuelve variantes en co-idiomas oficiales (`Alicante/Alacant` → `Alicante`, `Araba/Álava` → `Álava`, …).
3. Corrige errores de codificación **mojibake** detectados en algunos ficheros del INE (ej. `AndalucÃ­a` → `Andalucía`).
4. Aplica **fuzzy matching** (`difflib.get_close_matches`) como red de seguridad para variantes no cubiertas por las reglas anteriores.

Para las fuentes de ámbito **nacional** (`Euríbor`, `Spread`, `Ahorro`, `Mano de Obra`, `Materiales`) se usa `expandir_nacional_a_prov`, que replica el valor a las 52 provincias. Para las fuentes de ámbito **CCAA** (`IPC`) se usa `expandir_ccaa_a_prov`, que mapea la CCAA a sus provincias correspondientes.

### 2.3 Construcción del dataset maestro
1. Se genera un **esqueleto cartesiano** `Provincia × Trimestre` con `pd.MultiIndex.from_product` (todas las combinaciones posibles desde `2005-01-01`).
2. Cada fuente ya procesada se une con `pd.merge(how='left')`. Los trimestres o provincias sin dato quedan como `NaN`, preservando la integridad del esqueleto.
3. Tras la fusión se aplica una **normalización de ordenación alfabética hispana** (Álava/Ávila quedan en su posición correcta frente al orden ASCII por defecto de Python).

### 2.4 Decisión de diseño: Euríbor + Spread en vez de TAE (APRC)
Se excluye intencionadamente la **TAE (APRC)** para evitar **multicolinealidad perfecta** (`TAE ≈ Euríbor + Spread`), lo que degradaría la interpretabilidad de modelos como XGBoost y colapsaría la convergencia de modelos estadísticos como SARIMAX. Esta separación permite al modelo distinguir entre:
- El **coste macroeconómico del dinero** (Euríbor, dictado por el BCE).
- El **margen comercial de la banca privada** (Spread). 

## 1. Importación de Librerías
Importamos las librerías necesarias para la manipulación de datos (Pandas, Numpy), el manejo de expresiones regulares, y utilidades para comparaciones difusas (Fuzzy Matching) que nos ayudarán a estandarizar los nombres de las provincias.

In [1]:
import pandas as pd
import numpy as np
import os
import re
from difflib import get_close_matches
import unicodedata

pd.set_option('display.max_columns', None)


## 2. Configuración Geográfica y 'Mojibake'
En esta sección definimos nuestra lista oficial de 52 provincias. Además, desarrollamos una lógica robusta (la función `limpiar_provincia`) para tratar con los graves problemas de codificación de caracteres detectados en algunas fuentes del INE (conocido como *'mojibake'*), así como el mapeo estandarizado de provincias independientemente de la fuente de datos.

In [2]:
provincias_limpias = [
    'A Coruña', 'Albacete', 'Alicante', 'Almería', 'Badajoz', 'Barcelona', 'Burgos', 'Cantabria', 'Castellón', 'Ceuta', 'Ciudad Real', 'Comunidad Foral de Navarra', 'Comunidad de Madrid', 'Cuenca', 'Cáceres', 'Cádiz', 'Córdoba', 'Girona', 'Granada', 'Guadalajara', 'Guipúzcoa', 'Huelva', 'Huesca', 'Islas Baleares', 'Jaén', 'La Rioja', 'Las Palmas', 'León', 'Lleida', 'Lugo', 'Melilla', 'Málaga', 'Ourense', 'Palencia', 'Pontevedra', 'Principado de Asturias', 'Región de Murcia', 'Salamanca', 'Santa Cruz de Tenerife', 'Segovia', 'Sevilla', 'Soria', 'Tarragona', 'Teruel', 'Toledo', 'Valencia', 'Valladolid', 'Vizcaya', 'Zamora', 'Zaragoza', 'Álava', 'Ávila'
]

ccaa_a_prov = {
    'Andalucía': ['Almería', 'Cádiz', 'Córdoba', 'Granada', 'Huelva', 'Jaén', 'Málaga', 'Sevilla'],
    'Aragón': ['Huesca', 'Teruel', 'Zaragoza'],
    'Asturias': ['Principado de Asturias'],
    'Baleares': ['Islas Baleares'],
    'Canarias': ['Las Palmas', 'Santa Cruz de Tenerife'],
    'Cantabria': ['Cantabria'],
    'Castilla y León': ['Ávila', 'Burgos', 'León', 'Palencia', 'Salamanca', 'Segovia', 'Soria', 'Valladolid', 'Zamora'],
    'Castilla - La Mancha': ['Albacete', 'Ciudad Real', 'Cuenca', 'Guadalajara', 'Toledo'],
    'Cataluña': ['Barcelona', 'Girona', 'Lleida', 'Tarragona'],
    'Comunitat Valenciana': ['Alicante', 'Castellón', 'Valencia'],
    'Extremadura': ['Badajoz', 'Cáceres'],
    'Galicia': ['A Coruña', 'Lugo', 'Ourense', 'Pontevedra'],
    'Madrid': ['Comunidad de Madrid'],
    'Murcia': ['Región de Murcia'],
    'Navarra': ['Comunidad Foral de Navarra'],
    'País Vasco': ['Álava', 'Guipúzcoa', 'Vizcaya'],
    'Rioja': ['La Rioja'],
    'Ceuta': ['Ceuta'],
    'Melilla': ['Melilla']
}

def remove_accents_and_bizarre(input_str):
    s = str(input_str).lower()
    s = s.replace('ã', 'a').replace('³', 'o').replace('©', 'e').replace('±', 'n')
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z]', '', s)

clean_provs_map = {remove_accents_and_bizarre(p): p for p in provincias_limpias}

def limpiar_provincia(nombre):
    if pd.isna(nombre): return np.nan
    nombre_limpio = re.sub(r'^[0-9]+\s*', '', str(nombre)).strip()
    
    # Manejar agrupaciones
    if 'Alicante' in nombre_limpio or 'Alacant' in nombre_limpio: return 'Alicante'
    if 'Coru' in nombre_limpio: return 'A Coruña'
    if 'Asturias' in nombre_limpio: return 'Principado de Asturias'
    if 'Baleares' in nombre_limpio or 'Balears' in nombre_limpio: return 'Islas Baleares'
    if 'Castell' in nombre_limpio: return 'Castellón'
    if 'Valen' in nombre_limpio: return 'Valencia'
    if 'Navarra' in nombre_limpio: return 'Comunidad Foral de Navarra'
    if 'Rioja' in nombre_limpio: return 'La Rioja'
    if 'Madrid' in nombre_limpio: return 'Comunidad de Madrid'
    if 'Murcia' in nombre_limpio: return 'Región de Murcia'
    if 'Palmas' in nombre_limpio: return 'Las Palmas'
    if 'Almer' in nombre_limpio: return 'Almería'
    if 'cer' in nombre_limpio and 'C' in nombre_limpio: return 'Cáceres'
    if 'diz' in nombre_limpio and 'C' in nombre_limpio: return 'Cádiz'
    if 'rdo' in nombre_limpio and 'C' in nombre_limpio: return 'Córdoba'
    if 'Ja' in nombre_limpio and 'n' in nombre_limpio: return 'Jaén'
    if 'Le' in nombre_limpio and 'n' in nombre_limpio: return 'León'
    if 'M' in nombre_limpio and 'laga' in nombre_limpio: return 'Málaga'
    
    clean_target = remove_accents_and_bizarre(nombre_limpio)
    
    # Check exact substring match
    for k, v in clean_provs_map.items():
        # Requires at least 4 letters to avoid weird matches
        if len(clean_target) >= 4 and (k in clean_target or clean_target in k):
            return v
            
    matches = get_close_matches(clean_target, list(clean_provs_map.keys()), n=1, cutoff=0.5)
    if matches:
        return clean_provs_map[matches[0]]
    return nombre_limpio


## 3. Funciones de Transformación Temporal
Para poder cruzar todos los datos, hemos impuesto como condición estandarizar nuestro dataset a una base trimestral arrancando el **primer día de cada trimestre** (p. ej. `2005-01-01`). Las siguientes funciones se encargan de convertir distintos formatos originales a este estándar temporal ISO indispensable.

In [3]:

def q_to_date(ano, q):
    mes_q = (int(q) - 1) * 3 + 1
    return f"{ano}-{mes_q:02d}-01"

def mensual_a_trimestral(mes_str):
    mes_str = str(mes_str).strip()
    match = re.search(r'([0-9]{4}).*?([0-9]{1,2})', mes_str)
    if not match: return np.nan
    ano, mes = match.groups()
    trimestre = (int(mes) - 1) // 3 + 1
    return q_to_date(ano, trimestre)

def expandir_anual_a_trimestral(df, col_ano='Periodo'):
    df_expanded = []
    for _, row in df.iterrows():
        ano = str(row[col_ano]).strip()
        if not re.match(r'^[0-9]{4}$', ano): continue
        for q in [1, 2, 3, 4]:
            new_row = row.copy()
            new_row['Trimestre'] = q_to_date(ano, q)
            df_expanded.append(new_row)
    return pd.DataFrame(df_expanded).drop(columns=[col_ano])


## 4. Funciones de Imputación y Limpieza General
- `expandir_nacional_a_prov` y `expandir_ccaa_a_prov`: Permiten arrastrar e imputar el mismo dato general a un pequeño abanico de sub-provincias correspondientes.
- `limpiar_numero`: Arregla los problemas de notación numérica española con puntos/separadores de miles y decimales para asegurar que las variables siempre se lean como tipo float en Python.

In [4]:
def expandir_nacional_a_prov(df):
    df_exp = []
    for _, row in df.iterrows():
        for prov in provincias_limpias:
            new_row = row.copy()
            new_row['Provincia'] = prov
            df_exp.append(new_row)
    return pd.DataFrame(df_exp)

def expandir_ccaa_a_prov(df, col_ccaa='CCAA'):
    df_exp = []
    for _, row in df.iterrows():
        ccaa = str(row[col_ccaa]).strip()
        ccaa_limpia = re.sub(r'^[0-9]+\s*', '', ccaa).strip().lower()
        provs = []
        
        if 'andaluc' in ccaa_limpia: provs = ccaa_a_prov['Andalucía']
        elif 'arag' in ccaa_limpia: provs = ccaa_a_prov['Aragón']
        elif 'astur' in ccaa_limpia: provs = ccaa_a_prov['Asturias']
        elif 'balear' in ccaa_limpia or 'illes' in ccaa_limpia: provs = ccaa_a_prov['Baleares']
        elif 'canar' in ccaa_limpia: provs = ccaa_a_prov['Canarias']
        elif 'cantab' in ccaa_limpia: provs = ccaa_a_prov['Cantabria']
        elif 'mancha' in ccaa_limpia: provs = ccaa_a_prov['Castilla - La Mancha']
        elif 'castilla' in ccaa_limpia and 'le' in ccaa_limpia: provs = ccaa_a_prov['Castilla y León']
        elif 'catalu' in ccaa_limpia: provs = ccaa_a_prov['Cataluña']
        elif 'valenc' in ccaa_limpia or 'comunitat' in ccaa_limpia: provs = ccaa_a_prov['Comunitat Valenciana']
        elif 'extrem' in ccaa_limpia: provs = ccaa_a_prov['Extremadura']
        elif 'galic' in ccaa_limpia: provs = ccaa_a_prov['Galicia']
        elif 'madrid' in ccaa_limpia: provs = ccaa_a_prov['Madrid']
        elif 'murcia' in ccaa_limpia: provs = ccaa_a_prov['Murcia']
        elif 'navar' in ccaa_limpia: provs = ccaa_a_prov['Navarra']
        elif 'vasc' in ccaa_limpia or 'pais' in ccaa_limpia or 'pa' in ccaa_limpia and 's' in ccaa_limpia: provs = ccaa_a_prov['País Vasco']
        elif 'rioja' in ccaa_limpia: provs = ccaa_a_prov['Rioja']
        elif 'ceuta' in ccaa_limpia: provs = ccaa_a_prov['Ceuta']
        elif 'melilla' in ccaa_limpia: provs = ccaa_a_prov['Melilla']
        
        if not provs:
            for key, vals in ccaa_a_prov.items():
                if remove_accents_and_bizarre(key) in remove_accents_and_bizarre(ccaa) or remove_accents_and_bizarre(ccaa) in remove_accents_and_bizarre(key):
                    provs = vals
                    break
        if not provs: provs = [ccaa]
        
        for prov in provs:
            new_row = row.copy()
            new_row['Provincia'] = prov
            df_exp.append(new_row)
    return pd.DataFrame(df_exp).drop(columns=[col_ccaa])

def limpiar_numero(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip()
    if not val_str or val_str in ['.', '..']: return np.nan
    # Si viene con punto y es un numero pequeño (ej 3.915) en transmisiones podria ser miles. 
    # Generalmente el INE junta miles con punto y decimal con coma.
    if ',' in val_str:
        val_str = val_str.replace('.', '').replace(',', '.')
    else:
        # Check if the dot is used as decimal or thousands.
        # If it has a dot and no comma, is it english format or thousands?
        # If the number has 3 digits after dot, usually thousands in INE.
        if re.search(r'\.[0-9]{3}$', val_str):
            val_str = val_str.replace('.', '')
    try: return float(val_str)
    except: return np.nan


## 5. Carga y Procesamiento Iterativo de Datos
Comenzamos el bloque iterativo, cargando cada origen de datos (ficheros `.csv` o `.xlsx`) de forma independiente, aplicándole las funciones temporales correspondientes y las limpiezas de la variable Provincia desarrolladas en la sección anterior para homogeneizarlos antes de la gran unión.

In [5]:
dfs_a_unir = []

# Tinsa
df_tinsa = pd.read_csv('tinsa_prov_precios_trim.csv', engine='python', sep=',')
def tinsa_a_fecha(trim_str):
    match = re.search(r'([0-9]{4})\s*([1-4])T', str(trim_str))
    if not match: return np.nan
    return q_to_date(match.group(1), match.group(2))
df_tinsa['Trimestre'] = df_tinsa['Trimestre'].apply(tinsa_a_fecha)
df_tinsa['Provincia'] = df_tinsa['Provincia'].apply(limpiar_provincia)
df_tinsa = df_tinsa.groupby(['Provincia', 'Trimestre'], as_index=False)['Precio_m2'].mean()
dfs_a_unir.append(df_tinsa)

# Euribor
try:
    df_euribor = pd.read_csv('euribor_trim_fixed.csv', engine='python', sep=',')
    def q_str_a_fecha(trim_str):
        match = re.search(r'([0-9]{4})-Q([1-4])', str(trim_str))
        if not match: return np.nan
        return q_to_date(match.group(1), match.group(2))
    df_euribor['Trimestre'] = df_euribor['TIME_PERIOD'].apply(q_str_a_fecha)
    df_euribor = df_euribor[['Trimestre', 'EURIBOR_VALUE']].rename(columns={'EURIBOR_VALUE': 'Euribor'})
    dfs_a_unir.append(expandir_nacional_a_prov(df_euribor))
except Exception as e: print("Error Euribor:", e)

# Ahorro
try:
    df_ahorro = pd.read_csv('tasa_ahorro_trimestral_nac.csv', engine='python', sep=';')
    def q_str_a_fecha(trim_str):
        match = re.search(r'([0-9]{4})Q([1-4])', str(trim_str))
        if not match: return np.nan
        return q_to_date(match.group(1), match.group(2))
    df_ahorro['Trimestre'] = df_ahorro['Trimestre'].apply(q_str_a_fecha)
    df_ahorro = df_ahorro[['Trimestre', 'tasa_ahorro']]
    dfs_a_unir.append(expandir_nacional_a_prov(df_ahorro))
except Exception as e: print("Error Ahorro:", e)

# Spread
try:
    df_spread = pd.read_csv('spread_hipotecas_trim.csv', engine='python', sep=',')
    df_spread['Trimestre'] = pd.to_datetime(df_spread['Fecha']).apply(lambda x: q_to_date(x.year, x.quarter))
    df_spread = df_spread[['Trimestre', 'Diferencia']].rename(columns={'Diferencia': 'Spread_Hipotecas'})
    dfs_a_unir.append(expandir_nacional_a_prov(df_spread))
except Exception as e: print("Error Spread:", e)


In [6]:
# Hipotecas por Provincia
try:
    df_hip = pd.read_csv('hipotecas_por_prov_men.csv', sep=';', encoding='latin1', engine='python')
    df_hip['Trimestre'] = df_hip['Periodo'].apply(mensual_a_trimestral)
    df_hip['Provincia'] = df_hip['Provincias'].apply(limpiar_provincia)
    df_hip['Hipotecas_Total'] = df_hip['Total'].apply(limpiar_numero)
    dfs_a_unir.append(df_hip.groupby(['Provincia', 'Trimestre'])['Hipotecas_Total'].sum().reset_index())
    print(f"Hipotecas cargadas: {df_hip['Provincia'].nunique()} provincias.")
except Exception as e:
    print("Error Hipotecas:", e)

Hipotecas cargadas: 52 provincias.


In [7]:
# Población
try:
    df_pob = pd.read_csv('poblacion_por_prov_combinado.csv', sep=';', encoding='latin1', engine='python')
    col_prov = [c for c in df_pob.columns if 'prov' in c.lower()][0]
    df_pob['Provincia'] = df_pob[col_prov].apply(limpiar_provincia)
    df_pob['Poblacion'] = df_pob['Total'].apply(limpiar_numero)
    dfs_a_unir.append(expandir_anual_a_trimestral(df_pob[['Provincia', 'Periodo', 'Poblacion']]))
except Exception as e: print("Error Poblacion:", e)

# Edad
try:
    df_edad = pd.read_csv('edad_media_provincias_an.csv', sep=';', encoding='latin1', engine='python')
    df_edad['Provincia'] = df_edad['Provincias'].apply(limpiar_provincia)
    df_edad['Edad_Media'] = df_edad['Total'].apply(limpiar_numero)
    dfs_a_unir.append(expandir_anual_a_trimestral(df_edad[['Provincia', 'Periodo', 'Edad_Media']]))
except Exception as e: print("Error Edad:", e)

# PIB
try:
    df_pib = pd.read_excel('pib_pc_prov_an.xlsx')
    if 'Periodo' in df_pib.columns and 'Total' in df_pib.columns:
        df_pib['Provincia'] = df_pib[df_pib.columns[0]].apply(limpiar_provincia)
        df_pib['PIB_pc'] = df_pib['Total'].apply(limpiar_numero)
        dfs_a_unir.append(expandir_anual_a_trimestral(df_pib[['Provincia', 'Periodo', 'PIB_pc']]))
except Exception as e: print("Error PIB:", e)


In [8]:
# Mano Obra
try:
    df_mano = pd.read_csv('indice_mano_obra_men.csv', sep=';', encoding='latin1', engine='python')
    df_mano['Trimestre'] = df_mano['Periodo'].apply(mensual_a_trimestral)
    df_mano['Indice_Mano_Obra'] = df_mano['Total'].apply(limpiar_numero)
    dfs_a_unir.append(expandir_nacional_a_prov(df_mano.groupby('Trimestre')['Indice_Mano_Obra'].mean().reset_index()))
except Exception as e: print("Error Mano Obra:", e)

# Materiales
try:
    df_mat = pd.read_csv('indice_precio_materiales_men.csv', sep=';', encoding='latin1', engine='python')
    df_mat['Trimestre'] = df_mat['Periodo'].apply(mensual_a_trimestral)
    df_mat['Indice_Materiales'] = df_mat['Total'].apply(limpiar_numero)
    dfs_a_unir.append(expandir_nacional_a_prov(df_mat.groupby('Trimestre')['Indice_Materiales'].mean().reset_index()))
except Exception as e: print("Error Materiales:", e)

# IPC
try:
    df_ipc = pd.read_csv('ipc_ccaa_men.csv', sep=';', encoding='latin1', engine='python')
    col_ccaa = [c for c in df_ipc.columns if 'comunidad' in c.lower()][0]
    df_ipc['Trimestre'] = df_ipc['Periodo'].apply(mensual_a_trimestral)
    df_ipc['IPC'] = df_ipc['Total'].apply(limpiar_numero)
    df_ipc_trim = df_ipc.groupby([col_ccaa, 'Trimestre'])['IPC'].mean().reset_index()
    dfs_a_unir.append(expandir_ccaa_a_prov(df_ipc_trim, col_ccaa=col_ccaa))
except Exception as e: print("Error IPC:", e)

# EPA
try:
    df_epa = pd.read_csv('epa_prov.csv', sep=';', encoding='latin1', engine='python')
    def epa_a_fecha(trim_str):
        match = re.search(r'([0-9]{4})T([1-4])', str(trim_str))
        if not match: return np.nan
        return q_to_date(match.group(1), match.group(2))
    df_epa['Trimestre'] = df_epa['Periodo'].apply(epa_a_fecha)
    df_epa['Provincia'] = df_epa['Provincias'].apply(limpiar_provincia)
    df_epa['Tasa_Paro'] = df_epa['Total'].apply(limpiar_numero)
    dfs_a_unir.append(df_epa.groupby(['Provincia', 'Trimestre'])['Tasa_Paro'].mean().reset_index())
except Exception as e: print("Error EPA:", e)


## 6. Carga de Población Extranjera por Provincia (`pob_ext_provincias_2005_2025.csv`)

Esta fuente procede del **Padrón Municipal del INE** y recoge el número de residentes extranjeros empadronados en cada provincia española entre 2005 y 2025 con periodicidad **anual**. La variable resultante (`Pob_Ext`) sirve como proxy del dinamismo migratorio y de la presión demográfica sobre el mercado inmobiliario, dos factores especialmente relevantes en mercados como Málaga, Alicante o Madrid.

**Retos de limpieza detectados:**
- Los nombres de provincia suelen estar en formato invertido del INE (`Coruña, A`, `Rioja, La`, `Palmas, Las`) o en su variante de co-idioma oficial (`Araba/Álava`, `Bizkaia`, `Gipuzkoa`, `Alicante/Alacant`, `Castellón/Castelló`, `Valencia/València`, `Balears, Illes`).
- Los datos post-2022 de algunas provincias aparecen con el nombre en castellano antiguo (`Gerona`, `Lérida`, `Orense`, `Vizcaya`, `Guipúzcoa`, `Álava`) repetidos junto con la entrada ya existente en catalán/eusquera. Descartamos los duplicados manteniendo el nombre canónico del dataset.
- Los datos son **anuales** y se debe expandir cada observación a los cuatro trimestres del año correspondiente con `expandir_anual_a_trimestral`.

In [9]:
# Mapa de nombres INE -> nombre canónico del dataset
mapa_pob_ext = {
    # Nombres invertidos (formato INE)
    'coruña, a':              'A Coruña',
    'rioja, la':              'La Rioja',
    'palmas, las':            'Las Palmas',
    'balears, illes':         'Islas Baleares',
    # Variantes co-oficiales
    'araba/álava':            'Álava',
    'araba/alava':            'Álava',
    'bizkaia':                'Vizcaya',
    'gipuzkoa':               'Guipúzcoa',
    'alicante/alacant':       'Alicante',
    'castellón/castelló':     'Castellón',
    'castellon/castello':     'Castellón',
    'valencia/valència':      'Valencia',
    # Comunidades autónomas uniprovinciales
    'madrid':                 'Comunidad de Madrid',
    'navarra':                'Comunidad Foral de Navarra',
    'murcia':                 'Región de Murcia',
    'asturias':               'Principado de Asturias',
    # Nombres en castellano antiguo (duplicados post-2022)
    'gerona':                 'Girona',
    'lérida':                 'Lleida',
    'lerida':                 'Lleida',
    'orense':                 'Ourense',
    'vizcaya':                'Vizcaya',
    'guipúzcoa':              'Guipúzcoa',
    'guipuzcoa':              'Guipúzcoa',
    'álava':                  'Álava',
    'alava':                  'Álava',
}

try:
    df_pob_ext = pd.read_csv('pob_ext_provincias_2005_2025.csv', sep=';', encoding='utf-8')

    # Normalizar el campo provincia para aplicar el mapa
    df_pob_ext['prov_norm'] = df_pob_ext['provincia'].str.strip().str.lower()
    df_pob_ext['Provincia'] = df_pob_ext['prov_norm'].map(mapa_pob_ext).fillna(
        df_pob_ext['provincia'].str.strip()
    )

    # Eliminar duplicados (ej. Girona 2023 aparece también como 'Gerona')
    # Nos quedamos con la primera aparición para cada (Provincia, año)
    df_pob_ext = df_pob_ext.sort_values('provincia').drop_duplicates(
        subset=['Provincia', 'año'], keep='first'
    )

    df_pob_ext = df_pob_ext.rename(columns={'año': 'Periodo', 'pob_ext': 'Pob_Ext'})
    df_pob_ext['Periodo'] = df_pob_ext['Periodo'].astype(str)

    # Expandir datos anuales a trimestrales
    df_pob_ext_trim = expandir_anual_a_trimestral(
        df_pob_ext[['Provincia', 'Periodo', 'Pob_Ext']]
    )
    dfs_a_unir.append(df_pob_ext_trim)
    print(f"Población extranjera cargada: {df_pob_ext_trim.shape[0]} filas, "
          f"{df_pob_ext_trim['Provincia'].nunique()} provincias.")

except Exception as e:
    print("Error cargando pob_ext_provincias:", e)


Población extranjera cargada: 4368 filas, 52 provincias.


### Transmisiones de Propiedades por Provincia (`transmision_prov_trim_tot.csv`)

Esta fuente, también procedente del **Ministerio de Vivienda / Consejo General del Notariado**, contiene el número de transmisiones de propiedades inmobiliarias por provincia y trimestre desde 2005. Es una medida directa de la actividad compradora y refleja el pulso del mercado inmobiliario local.

A diferencia de la versión anterior, el fichero actual ya viene **agregado por trimestre** (`año`, `trimestre`, `provincia`, `transmisiones`), lo que simplifica su integración: únicamente hay que convertir el par `(año, trimestre)` a la fecha ISO estándar `YYYY-MM-DD` con la función auxiliar `q_to_date` ya definida.

Los nombres de provincia siguen el mismo esquema que el Padrón Municipal, por lo que se aplica el mismo diccionario de estandarización (`mapa_pob_ext`).

In [10]:
# Transmisiones de Propiedades — nueva fuente trimestral
try:
    df_transm = pd.read_csv('transmision_prov_trim_tot.csv', sep=';', encoding='utf-8')

    # Mapear nombres de provincia al estándar canónico del dataset
    df_transm['prov_norm'] = df_transm['provincia'].str.strip().str.lower()
    df_transm['Provincia'] = df_transm['prov_norm'].map(mapa_pob_ext).fillna(
        df_transm['provincia'].str.strip()
    )

    # Construir la fecha trimestral ISO a partir de las columnas año + trimestre
    df_transm['Trimestre'] = df_transm.apply(
        lambda r: q_to_date(int(r['año']), int(r['trimestre'])), axis=1
    )

    df_transm = (
        df_transm
        .groupby(['Provincia', 'Trimestre'])['transmisiones']
        .sum()
        .reset_index()
        .rename(columns={'transmisiones': 'Transmisiones_Total'})
    )
    dfs_a_unir.append(df_transm)
    print(f"Transmisiones cargadas: {len(df_transm)} filas, "
          f"{df_transm['Provincia'].nunique()} provincias.")
except Exception as e:
    print("Error cargando transmision_prov_trim_tot:", e)


Transmisiones cargadas: 4312 filas, 52 provincias.


## 6. Fusión del Dataset Maestro (`pd.merge`)
Con todo nuestro conjunto de DataFrames parciales acumulados en la lista `dfs_a_unir`, la lógica ahora consiste en determinar cuál es el espectro temporal común deseado (a partir de `2005-01-01` en adelante).
Configurada la base `Provincia` x `Trimestre`, empezamos a completar iterativamente la matriz `df_master` mediante fusiones de izquierda (`pd.merge(how='left')`). Si el dataset bruto de origen carecía de dicho periodo histórico o provincia, reflejará de forma natural un valor nulo (`NaN`) manteniendo la integridad y alienamiento correcto del resto de variables del CSV.

In [11]:
trimestres_all = sorted(list(set().union(*(df['Trimestre'].dropna().unique() for df in dfs_a_unir))))
trimestres_clean = [t for t in trimestres_all if str(t).startswith('20') and str(t) >= '2005-01-01']

base = pd.MultiIndex.from_product([provincias_limpias, trimestres_clean], names=['Provincia', 'Trimestre']).to_frame(index=False)
df_master = base.copy()

for i, dfx in enumerate(dfs_a_unir):
    # Asegurarnos de que no hay NaN en Provincia
    dfx = dfx.dropna(subset=['Provincia', 'Trimestre'])
    
    cols_to_agg = [c for c in dfx.columns if c not in ['Provincia', 'Trimestre']]
    df_agg = dfx.groupby(['Provincia', 'Trimestre'])[cols_to_agg].mean().reset_index()
    df_master = pd.merge(df_master, df_agg, on=['Provincia', 'Trimestre'], how='left')

print(f"Dimensiones del Dataset Final: {df_master.shape}")
df_master.head()


Dimensiones del Dataset Final: (4368, 16)


,Provincia,Trimestre,Precio_m2,Euribor,tasa_ahorro,Spread_Hipotecas,Hipotecas_Total,Poblacion,Edad_Media,PIB_pc,Indice_Mano_Obra,Indice_Materiales,IPC,Tasa_Paro,Pob_Ext,Transmisiones_Total
0,A Coruña,2005-01-01,1070.856714,2.319097,0.1,1.110903,7527.0,1126707.0,43.37,18336.0,328.903333,1460.814286,3.300000,10.58,23855.0,3179.0
1,A Coruña,2005-04-01,1102.888987,2.185862,16.7,1.207472,6743.0,1126707.0,43.37,18336.0,335.170000,1502.021429,3.100000,10.68,23855.0,4076.0
2,A Coruña,2005-07-01,1137.313846,2.204288,7.2,1.082379,6372.0,1126707.0,43.37,18336.0,335.360000,1569.828571,3.433333,8.20,23855.0,4332.0
3,A Coruña,2005-10-01,1172.538976,2.627969,9.7,0.745365,6425.0,1126707.0,43.37,18336.0,339.743333,1610.842857,3.533333,9.07,23855.0,4203.0
4,A Coruña,2006-01-01,1205.433802,2.954492,-2.1,0.808841,7171.0,1129141.0,43.62,19678.0,339.833333,1674.630952,4.000000,9.55,25468.0,4671.0


In [12]:
df_master.tail()

,Provincia,Trimestre,Precio_m2,Euribor,tasa_ahorro,Spread_Hipotecas,Hipotecas_Total,Poblacion,Edad_Media,PIB_pc,Indice_Mano_Obra,Indice_Materiales,IPC,Tasa_Paro,Pob_Ext,Transmisiones_Total
4363,Ávila,2024-10-01,989.168120,2.547879,17.1,0.618788,492.0,160463.0,48.46,24928.0,470.476667,2905.295238,2.133333,9.47,16969.0,3805.0
4364,Ávila,2025-01-01,1021.998740,2.445531,4.9,0.481135,414.0,160738.0,48.51,NaN,473.823333,2976.490476,2.533333,8.35,17853.0,3630.0
4365,Ávila,2025-04-01,1019.785520,2.101523,20.4,0.695144,435.0,160738.0,48.51,NaN,NaN,NaN,2.100000,11.89,17853.0,11076.0
4366,Ávila,2025-07-01,1037.656570,2.120970,4.6,0.602364,401.0,160738.0,48.51,NaN,NaN,NaN,2.766667,7.08,17853.0,12816.0
4367,Ávila,2025-10-01,1047.329133,2.224030,NaN,0.449303,555.0,160738.0,48.51,NaN,NaN,NaN,2.966667,8.20,17853.0,2610.0


## 7. Fix de Ordenación Alfabética (Álava / Ávila)
En la lexicografía española, las provincias que comienzan por vocales con tilde (como **Álava** o **Ávila**) deben aparecer al principio de la lista, siguiendo el orden de la letra base 'A'. Sin embargo, por defecto, Python y Pandas utilizan el orden ASCII/Unicode, lo que relega estas provincias al final del dataset (después de Zaragoza).

Para solucionar esto de forma técnica y profesional para el TFM, aplicamos una **normalización de caracteres**:
1. Creamos una columna temporal `Provincia_Sort`.
2. Eliminamos las tildes únicamente para el cálculo del orden.
3. Ordenamos el DataFrame por esta columna auxiliar.
4. Eliminamos la columna auxiliar para mantener el dataset limpio.

In [13]:
# Función para normalizar nombres (quitar tildes para ordenación)
import unicodedata

def normalizar_para_orden(texto):
    if pd.isna(texto): return texto
    # Normalizamos a NFD y filtramos caracteres de combinación (tildes)
    return ''.join(c for c in unicodedata.normalize('NFD', str(texto)) 
                  if unicodedata.category(c) != 'Mn').lower()

# Aplicar el fix de ordenación
df_master['Provincia_Sort'] = df_master['Provincia'].apply(normalizar_para_orden)
df_master = df_master.sort_values(by=['Provincia_Sort', 'Trimestre']).drop(columns=['Provincia_Sort']).reset_index(drop=True)

# Guardar el CSV final con el orden correcto
df_master.to_csv('dataset_final_tfm.csv', index=False, encoding='utf-8')
df_master.head(10)



,Provincia,Trimestre,Precio_m2,Euribor,tasa_ahorro,Spread_Hipotecas,Hipotecas_Total,Poblacion,Edad_Media,PIB_pc,Indice_Mano_Obra,Indice_Materiales,IPC,Tasa_Paro,Pob_Ext,Transmisiones_Total
0,A Coruña,2005-01-01,1070.856714,2.319097,0.1,1.110903,7527.00,1126707.0,43.37,18336.0,328.903333,1460.814286,3.300000,10.58,23855.0,3179.0
1,A Coruña,2005-04-01,1102.888987,2.185862,16.7,1.207472,6743.00,1126707.0,43.37,18336.0,335.170000,1502.021429,3.100000,10.68,23855.0,4076.0
2,A Coruña,2005-07-01,1137.313846,2.204288,7.2,1.082379,6372.00,1126707.0,43.37,18336.0,335.360000,1569.828571,3.433333,8.20,23855.0,4332.0
3,A Coruña,2005-10-01,1172.538976,2.627969,9.7,0.745365,6425.00,1126707.0,43.37,18336.0,339.743333,1610.842857,3.533333,9.07,23855.0,4203.0
4,A Coruña,2006-01-01,1205.433802,2.954492,-2.1,0.808841,7171.00,1129141.0,43.62,19678.0,339.833333,1674.630952,4.000000,9.55,25468.0,4671.0
5,A Coruña,2006-04-01,1249.906651,3.315645,14.9,0.757688,7502.00,1129141.0,43.62,19678.0,346.220000,1785.590476,4.100000,7.93,25468.0,4360.0
6,A Coruña,2006-07-01,1285.271595,3.622785,5.6,0.783882,5830.00,1129141.0,43.62,19678.0,345.426667,1801.171429,3.533333,6.88,25468.0,4454.0
7,A Coruña,2006-10-01,1321.852208,3.858413,9.2,0.834921,1301.82,1129141.0,43.62,19678.0,347.096667,1739.578571,2.500000,8.13,25468.0,4854.0
8,A Coruña,2007-01-01,1357.974781,4.087656,-4.2,0.829010,7135.00,1132792.0,43.87,21260.0,346.793333,1719.126190,2.066667,9.53,28137.0,15791.0
9,A Coruña,2007-04-01,1399.518174,4.381048,14.0,0.738952,6385.00,1132792.0,43.87,21260.0,353.406667,1829.664286,2.000000,8.22,28137.0,9520.0


In [14]:
df_master.tail()

,Provincia,Trimestre,Precio_m2,Euribor,tasa_ahorro,Spread_Hipotecas,Hipotecas_Total,Poblacion,Edad_Media,PIB_pc,Indice_Mano_Obra,Indice_Materiales,IPC,Tasa_Paro,Pob_Ext,Transmisiones_Total
4363,Zaragoza,2024-10-01,1597.598649,2.547879,17.1,0.618788,3483.00,987763.0,45.2,37578.0,470.476667,2905.295238,2.600000,7.20,160739.0,10427.0
4364,Zaragoza,2025-01-01,1628.892950,2.445531,4.9,0.481135,3365.00,998443.0,45.3,NaN,473.823333,2976.490476,2.833333,8.37,164324.0,5678.0
4365,Zaragoza,2025-04-01,1670.180148,2.101523,20.4,0.695144,2187.04,998443.0,45.3,NaN,NaN,NaN,2.366667,7.86,164324.0,13382.0
4366,Zaragoza,2025-07-01,1718.407413,2.120970,4.6,0.602364,3785.00,998443.0,45.3,NaN,NaN,NaN,2.900000,9.02,164324.0,12645.0
4367,Zaragoza,2025-10-01,1773.141714,2.224030,NaN,0.449303,3079.00,998443.0,45.3,NaN,NaN,NaN,3.033333,8.46,164324.0,4722.0
